[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aurelio-labs/semantic-router/blob/main/docs/encoders/mistral-encoder.ipynb) [![Open nbviewer](https://raw.githubusercontent.com/pinecone-io/examples/master/assets/nbviewer-shield.svg)](https://nbviewer.org/github/aurelio-labs/semantic-router/blob/main/docs/encoders/mistral-encoder.ipynb)

# Using Mistral Models

The Mistral embedding model (`mistral-embed`) can be used with our `MistralEncoder` and usage is primarily the same as with other embedding models. The model produces high-quality embeddings for semantic search, classification, and other text similarity tasks.

## Getting Started

We start by installing semantic-router. `MistralEncoder` is part of the base install:

In [ ]:
!pip install -qU "semantic-router>=1.0.0"

We start by defining a route: a name, and some example phrases (utterances) that should trigger it.

In [ ]:
from semantic_router import Route

politics = Route(
    name="politics",
    utterances=[
        "isn't politics the best thing ever",
        "why don't you tell me about your political opinions",
        "don't you just love the president",
        "don't you just hate the president",
        "they're going to destroy this country!",
        "they will save the country!",
    ],
)

Let's define another for good measure:

In [ ]:
chitchat = Route(
    name="chitchat",
    utterances=[
        "how's the weather today?",
        "how are things going?",
        "lovely weather today",
        "the weather is horrendous",
        "let's go to the chippy",
    ],
)

routes = [politics, chitchat]

Now we initialize our embedding model, we will use the `mistral-embed` model.

In [ ]:
import os
from getpass import getpass
from semantic_router.encoders import MistralEncoder

os.environ["MISTRAL_API_KEY"] = os.getenv("MISTRAL_API_KEY") or getpass(
    "Enter Mistral API Key: "
)

encoder = MistralEncoder(name="mistral-embed")

We can call the encoder directly to embed a few texts:

In [ ]:
embeds = encoder(
    docs=[
        "I'm interested in learning about llama 2",
        "Don't you love politics?",
        "How's the weather today?",
        "I love the politics",
    ]
)
len(embeds), len(embeds[0])

Now we define the router. `IndexedRouter` embeds every utterance with our encoder and stores it in an index (an in-memory `LocalIndex` by default). When called, the router consumes text (a query) and outputs the route it belongs to.

In [ ]:
from semantic_router import IndexedRouter

router = IndexedRouter(encoder=encoder, routes=routes)

We can check the dimensionality of our vectors, and how many are stored, with `describe()` on the router's index:

In [ ]:
router.index.describe()

Now we can test it:

In [ ]:
router("don't you love politics?")

In [ ]:
router("how's the weather today?")

Both should be classified accurately. Each `RouteChoice` also carries a `confidence`. `calibrated=False` tells us it is the raw similarity between the query and the route's utterances, not a probability.

What if we send a query that is unrelated to our existing routes?

In [ ]:
router("I'm interested in learning about llama 2")

We still get a route. A router with no `confidence_threshold` always returns its top route, however low the confidence. To see every route ranked, pass `limit=None`:

In [ ]:
router("I'm interested in learning about llama 2", limit=None)

To get no route for queries like this, set a `confidence_threshold`: a route is only returned when its confidence clears it. Raw similarities sit on a different scale for every encoder, so check the scores of a few real and unrelated queries with Mistral before picking a bar:

```python
router.confidence_threshold = ...  # between your unrelated and real scores
```

Better still, calibrate the router with `fit()` on a few held-out labelled queries, so that a confidence of `0.8` means the router is right about 80% of the time. See [route calibration](https://github.com/aurelio-labs/semantic-router/blob/main/docs/06-calibration.ipynb).

---